In [ ]:
!pip install --user regex sentence-transformers

In [13]:
from sentence_transformers import SentenceTransformer 
import numpy as np

name = 'sergeyzh/rubert-large-uncased-query'
model = SentenceTransformer(name)
print("Модель"+ str({name}) + "загружена")
print({model.get_sentence_embedding_dimension()})

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 289.72it/s]


Модель{'sergeyzh/rubert-large-uncased-query'}загружена
{1024}


C:\Users\User\AppData\Local\Temp\ipykernel_7940\2429095034.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print({model.get_sentence_embedding_dimension()})


In [14]:
def get_embedding(text: str, model, max_len = 512):
    chunks = [text[i:i+max_len*3] for i in range(0, len(text), max_len*3)] # max_len*3 ~ 1536
    if not(chunks):
        return np.zeros(1024)
    chunk_embeddings = model.encode(chunks, convert_to_numpy=True) 
    medium = np.mean(chunk_embeddings, axis = 0) # усреднение
    medium /= np.linalg.norm(medium) # делим на длину вектора
    return medium

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

def duplicats(new_doc: str, exists: list, model, threshold=0.95):
    new_embedding = get_embedding(new_doc, model)
    duplicates = []
    for doc in exists:
        exist_doc = get_embedding(doc['text'], model)
        sim = cosine_similarity([new_embedding], [exist_doc])[0][0] # схожесть косинусов

        if sim >= threshold:
            duplicates.append({'id': doc['id'],
                               'similarity': sim })
    duplicates.sort(key=lambda x : x['similarity'], reverse=True)
    return duplicates

In [17]:
#test
doc1_text = "Истец просит взыскать задолженность по договору займа в размере 50 000 рублей."
doc2_text = "Гражданин подал иск о взыскании долга по договору займа на сумму 50 тысяч рублей."
doc3_text = "Ответчик возражает против удовлетворения требований о расторжении брака."

existing_documents = [
    {'id': 1, 'text': doc1_text},
    {'id': 3, 'text': doc3_text}
]
duplicates = duplicats(doc2_text, existing_documents, model)
if duplicates:
    for d in duplicates:
        print(f" Найден дубликат ID {d['id']} со сходством {d['similarity']:.4f}")
else:
    print("Дубликатов не найдено.")

 Найден дубликат ID 1 со сходством 0.9529
